# 금융 상담 라벨 JSON → DataFrame 전용 노트북

이 노트북은 `2.데이터(NIA)` 폴더 바로 아래에 두고 실행하는 것을 기준으로 합니다.

- 읽는 범위: `Training/02.라벨링데이터`, `Validation/02.라벨링데이터`
- 제외 범위: `01.원천데이터`, `TS_*`, `VS_*`, ZIP 파일
- 행 단위: `qa_data` 항목 1개 = DataFrame 1행
- 중요 경계: Training은 학습·RAG 색인용, Validation은 평가용이며 서로 섞지 않습니다.
- 상담 본문과 생성 산출물은 개인정보 가능성이 있으므로 Git에 올리지 않습니다.


## 1. 환경과 실행 모드

처음에는 `RUN_MODE = 'sample'`로 구조를 확인하세요. 정상 확인 후 `full`로 바꾸면 90,000개 라벨 JSON 전체를 읽습니다.

노트북이 데이터 루트 밖에 있다면 환경변수 `FINANCIAL_DATA_ROOT`에 `2.데이터(NIA)` 경로를 지정할 수 있습니다.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, Mapping

import pandas as pd
from IPython.display import display

RUN_MODE = 'sample'          # 'sample' 또는 'full'
SAMPLE_FILES_PER_DOMAIN = 100
SAVE_OUTPUTS = True
PROGRESS_EVERY = 5000

DATA_ROOT = Path(os.environ.get('FINANCIAL_DATA_ROOT', Path.cwd())).expanduser().resolve()
TRAINING_ROOT = DATA_ROOT / 'Training' / '02.라벨링데이터'
VALIDATION_ROOT = DATA_ROOT / 'Validation' / '02.라벨링데이터'
OUTPUT_DIR = DATA_ROOT / 'processed_financial_dataframe'

if RUN_MODE not in {'sample', 'full'}:
    raise ValueError("RUN_MODE는 'sample' 또는 'full'이어야 합니다.")

missing = [path for path in (TRAINING_ROOT, VALIDATION_ROOT) if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        '라벨링데이터 폴더를 찾지 못했습니다. 노트북을 2.데이터(NIA) 폴더에 두거나 '
        f'FINANCIAL_DATA_ROOT를 지정하세요: {missing}'
    )

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'RUN_MODE: {RUN_MODE}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


## 2. 라벨 파일 탐색

도메인 폴더(`TL_보험`, `TL_은행`, `TL_증권`, `VL_*`)별로 JSON을 찾습니다. 샘플 모드는 각 도메인 앞부분만 읽어 폴더·스키마가 맞는지 빠르게 확인합니다.


In [ ]:
def nfc(value: str) -> str:
    return unicodedata.normalize('NFC', value)


def discover_label_files(root: Path, *, limit_per_domain: int | None) -> list[Path]:
    files: list[Path] = []
    domain_dirs = sorted(
        (path for path in root.iterdir() if path.is_dir() and nfc(path.name).startswith(('TL_', 'VL_'))),
        key=lambda path: nfc(path.name),
    )
    for domain_dir in domain_dirs:
        domain_files = sorted(path for path in domain_dir.rglob('*.json') if path.is_file())
        if limit_per_domain is not None:
            domain_files = domain_files[:limit_per_domain]
        files.extend(domain_files)
    return files


limit = SAMPLE_FILES_PER_DOMAIN if RUN_MODE == 'sample' else None
training_files = discover_label_files(TRAINING_ROOT, limit_per_domain=limit)
validation_files = discover_label_files(VALIDATION_ROOT, limit_per_domain=limit)

inventory = pd.DataFrame(
    [
        {'split': 'Training', 'selected_json_files': len(training_files)},
        {'split': 'Validation', 'selected_json_files': len(validation_files)},
    ]
)
display(inventory)

if not training_files or not validation_files:
    raise RuntimeError('Training 또는 Validation 라벨 JSON을 찾지 못했습니다.')


## 3. JSON 평탄화와 DataFrame 생성

중첩 딕셔너리는 `_`로 연결해 열 이름을 만듭니다. 예: `source.source_id` → `source_source_id`, `qa_data[].input.question` → `qa_input_question`.

한 JSON 안에 QA가 여러 개 있어도 버리지 않고 모두 행으로 보존하되, 검증 단계에서 현재 계약 위반으로 표시합니다.


In [ ]:
def flatten_mapping(value: Mapping[str, Any], *, prefix: str = '') -> dict[str, Any]:
    flattened: dict[str, Any] = {}
    for key, child in value.items():
        column = f'{prefix}_{key}' if prefix else str(key)
        if isinstance(child, Mapping):
            flattened.update(flatten_mapping(child, prefix=column))
        elif isinstance(child, list):
            flattened[column] = json.dumps(child, ensure_ascii=False)
        else:
            flattened[column] = child
    return flattened


def path_metadata(path: Path) -> tuple[str, str]:
    parts = [nfc(part) for part in path.parts]
    split = 'Training' if 'Training' in parts else 'Validation'
    marker = 'TL_' if split == 'Training' else 'VL_'
    domain_part = next((part for part in parts if part.startswith(marker)), '')
    domain = domain_part.split('_', 1)[1] if '_' in domain_part else ''
    return split, domain


def decode_json(payload: bytes) -> dict[str, Any]:
    item = json.loads(payload.decode('utf-8-sig'))
    if not isinstance(item, dict):
        raise TypeError('JSON 최상위 값은 object여야 합니다.')
    return item


def load_financial_labels(files: list[Path], *, data_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    issues: list[dict[str, Any]] = []

    for file_number, path in enumerate(files, start=1):
        try:
            payload = path.read_bytes()
            item = decode_json(payload)
            qa_data = item.get('qa_data')
            if not isinstance(qa_data, list):
                raise TypeError('qa_data가 list가 아닙니다.')

            split, domain = path_metadata(path)
            source_file = path.relative_to(data_root).as_posix()
            base = flatten_mapping({key: value for key, value in item.items() if key != 'qa_data'})

            if len(qa_data) != 1:
                issues.append({
                    'source_file': source_file,
                    'issue_type': 'qa_count_contract',
                    'detail': f'qa_data length={len(qa_data)}',
                })

            document_rows: list[dict[str, Any]] = []
            for qa_index, qa_item in enumerate(qa_data):
                if not isinstance(qa_item, dict):
                    raise TypeError(f'qa_data[{qa_index}]가 object가 아닙니다.')
                row = dict(base)
                row.update(flatten_mapping(qa_item, prefix='qa'))
                row.update({
                    '_split': split,
                    '_domain': domain,
                    '_qa_index': qa_index,
                    '_source_file': source_file,
                    '_source_sha256': hashlib.sha256(payload).hexdigest(),
                })
                document_rows.append(row)
            rows.extend(document_rows)
        except (OSError, UnicodeDecodeError, json.JSONDecodeError, TypeError) as error:
            issues.append({
                'source_file': str(path),
                'issue_type': type(error).__name__,
                'detail': str(error),
            })

        if PROGRESS_EVERY and file_number % PROGRESS_EVERY == 0:
            print(f'{file_number:,}/{len(files):,} JSON 처리 완료')

    dataframe = pd.DataFrame(rows)
    for column in ('_split', '_domain'):
        if column in dataframe.columns:
            dataframe[column] = dataframe[column].astype('category')
    return dataframe, pd.DataFrame(issues)


selected_files = training_files + validation_files
financial_df, issues_df = load_financial_labels(selected_files, data_root=DATA_ROOT)
print(f'DataFrame: {financial_df.shape[0]:,}행 × {financial_df.shape[1]:,}열')
print(f'검토 필요 항목: {len(issues_df):,}건')
display(financial_df.head(3))


## 4. 데이터 계약 검증

전체 실행 시 기대 행 수는 Training 80,000행, Validation 10,000행입니다. `source_id` 3개가 두 split에 겹치는 것은 기존 감사에서 확인된 경고이며, 모델 선택·최종 평가 전에 해당 source의 13개 QA 행을 격리해야 합니다.


In [ ]:
REQUIRED_COLUMNS = {
    'source_source_id',
    'source_source_institution',
    'consulting_consulting_category',
    'consulting_consulting_topic',
    'qa_qa_id',
    'qa_instruction',
    'qa_input_question',
    'qa_input_answer',
    'qa_input_follow_up_question',
    'qa_output',
    '_split',
    '_domain',
}

missing_columns = sorted(REQUIRED_COLUMNS - set(financial_df.columns))
duplicate_qa_rows = int(financial_df['qa_qa_id'].duplicated(keep=False).sum()) if 'qa_qa_id' in financial_df else None

training_source_ids = set(
    financial_df.loc[financial_df['_split'] == 'Training', 'source_source_id'].dropna().astype(str)
)
validation_source_ids = set(
    financial_df.loc[financial_df['_split'] == 'Validation', 'source_source_id'].dropna().astype(str)
)
source_overlap = sorted(training_source_ids & validation_source_ids)

summary = {
    'run_mode': RUN_MODE,
    'rows': int(len(financial_df)),
    'columns': int(len(financial_df.columns)),
    'rows_by_split': financial_df['_split'].value_counts().sort_index().to_dict(),
    'rows_by_domain': financial_df['_domain'].value_counts().sort_index().to_dict(),
    'missing_required_columns': missing_columns,
    'duplicate_qa_id_rows': duplicate_qa_rows,
    'cross_split_source_id_overlap': len(source_overlap),
    'cross_split_source_id_examples': source_overlap[:20],
    'issues': int(len(issues_df)),
}
display(pd.Series(summary, name='value').to_frame())

if RUN_MODE == 'full':
    expected_split = {'Training': 80000, 'Validation': 10000}
    expected_domain = {'보험': 27000, '은행': 45000, '증권': 18000}
    actual_split = {str(key): int(value) for key, value in financial_df['_split'].value_counts().items()}
    actual_domain = {str(key): int(value) for key, value in financial_df['_domain'].value_counts().items()}
    if actual_split != expected_split:
        raise AssertionError(f'split 행 수 불일치: {actual_split}')
    if actual_domain != expected_domain:
        raise AssertionError(f'domain 행 수 불일치: {actual_domain}')

if missing_columns or duplicate_qa_rows or not issues_df.empty:
    print('검증 경고: 저장 전에 missing columns, QA 중복, issues_df를 확인하세요.')
else:
    print('기본 스키마·QA ID 검증 통과')


## 5. 금융 상담에 특화된 DataFrame 뷰

- `train_df`, `validation_df`: split을 보존한 원본 평탄화 뷰
- `classification_df`: 은행/보험/증권 및 상담 주제 분류 실험용
- `rag_df`: 질문 영역과 정답 영역을 분리한 RAG 입력용

Validation의 정답 또는 문서를 Training/RAG 색인에 넣으면 평가 누수가 발생하므로 `purpose`를 반드시 지킵니다.


In [ ]:
def clean_text(value: Any) -> str:
    if value is None or (not isinstance(value, (list, dict)) and pd.isna(value)):
        return ''
    return str(value).replace('\u2028', '\n').replace('\u2029', '\n').strip()


def join_fields(row: pd.Series, fields: list[tuple[str, str]]) -> str:
    parts = []
    for column, label in fields:
        text = clean_text(row.get(column))
        if text:
            parts.append(f'{label}: {text}')
    return '\n'.join(parts)


train_df = financial_df.loc[financial_df['_split'] == 'Training'].copy()
validation_df = financial_df.loc[financial_df['_split'] == 'Validation'].copy()

classification_df = financial_df[[
    '_split', '_domain', 'source_source_id', 'qa_qa_id',
    'consulting_consulting_category', 'consulting_consulting_topic',
    'qa_qa_topic', 'qa_consulting_purpose',
    'qa_instruction', 'qa_input_question', 'qa_input_follow_up_question',
]].copy()
classification_df['model_input_text'] = financial_df.apply(
    lambda row: join_fields(row, [
        ('qa_instruction', '요구사항'),
        ('qa_input_question', '고객 질문'),
        ('qa_input_follow_up_question', '꼬리 질문'),
    ]),
    axis=1,
)

rag_df = financial_df[[
    '_split', '_domain', 'source_source_id', 'qa_qa_id',
    'consulting_consulting_category', 'consulting_consulting_topic',
    'qa_qa_topic', 'qa_consulting_purpose', '_source_file', '_source_sha256',
]].copy()
rag_df['purpose'] = rag_df['_split'].map({'Training': 'index', 'Validation': 'evaluation'})
rag_df['retrieval_text'] = classification_df['model_input_text']
rag_df['answer_text'] = financial_df.apply(
    lambda row: join_fields(row, [
        ('qa_input_answer', '상담사 답변'),
        ('qa_output', '종합 답변'),
    ]),
    axis=1,
)

insurance_df = financial_df.loc[financial_df['_domain'] == '보험'].copy()
bank_df = financial_df.loc[financial_df['_domain'] == '은행'].copy()
securities_df = financial_df.loc[financial_df['_domain'] == '증권'].copy()

display(classification_df.head(3))
display(rag_df.head(3))


## 6. 기본 EDA

평균 같은 숫자 하나보다 split·도메인·상담 주제의 분포와 결측을 먼저 확인합니다. 상담 텍스트 길이는 문자 수 기준의 탐색값이며 모델 토큰 수와 같지 않습니다.


In [ ]:
eda_counts = (
    financial_df.groupby(['_split', '_domain'], observed=True)
    .size()
    .rename('rows')
    .reset_index()
)
display(eda_counts)

topic_counts = (
    financial_df['qa_qa_topic']
    .value_counts(dropna=False)
    .rename_axis('qa_topic')
    .reset_index(name='rows')
)
display(topic_counts.head(20))

important_columns = [
    'source_client_gender', 'source_client_age', 'source_consulting_client',
    'source_consulting_client_type', 'consulting_consulting_topic',
    'qa_qa_topic', 'qa_consulting_purpose', 'qa_input_question',
    'qa_input_answer', 'qa_input_follow_up_question', 'qa_output',
]
available_columns = [column for column in important_columns if column in financial_df.columns]
null_report = (
    financial_df[available_columns].isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename('missing_percent')
    .to_frame()
)
display(null_report)

text_length_report = pd.DataFrame({
    'question_chars': financial_df['qa_input_question'].map(clean_text).str.len(),
    'answer_chars': financial_df['qa_input_answer'].map(clean_text).str.len(),
    'output_chars': financial_df['qa_output'].map(clean_text).str.len(),
}).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
display(text_length_report)


## 7. 검증된 산출물 저장

전체 텍스트를 CSV/Excel로 저장하면 크기·인코딩·민감정보 노출 위험이 커집니다. 전체본은 Parquet, 사람 확인용은 앞부분 Excel 샘플, RAG 입력은 JSONL로 저장합니다.


In [ ]:
def write_jsonl(dataframe: pd.DataFrame, path: Path) -> None:
    dataframe.to_json(path, orient='records', lines=True, force_ascii=False)


if SAVE_OUTPUTS:
    try:
        import pyarrow  # noqa: F401
        import openpyxl  # noqa: F401
    except ImportError as error:
        raise ImportError('저장을 위해 pandas pyarrow openpyxl을 설치하세요.') from error

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    suffix = 'sample' if RUN_MODE == 'sample' else 'full'

    output_paths = {
        'all_parquet': OUTPUT_DIR / f'financial_qa_all_{suffix}.parquet',
        'training_parquet': OUTPUT_DIR / f'training_qa_{suffix}.parquet',
        'validation_parquet': OUTPUT_DIR / f'validation_qa_{suffix}.parquet',
        'classification_parquet': OUTPUT_DIR / f'classification_view_{suffix}.parquet',
        'training_rag_jsonl': OUTPUT_DIR / f'training_rag_index_{suffix}.jsonl',
        'validation_rag_jsonl': OUTPUT_DIR / f'validation_rag_evaluation_{suffix}.jsonl',
        'sample_excel': OUTPUT_DIR / f'financial_qa_preview_{suffix}.xlsx',
        'validation_report': OUTPUT_DIR / f'validation_report_{suffix}.json',
        'issues': OUTPUT_DIR / f'issues_{suffix}.jsonl',
    }

    financial_df.to_parquet(output_paths['all_parquet'], index=False)
    train_df.to_parquet(output_paths['training_parquet'], index=False)
    validation_df.to_parquet(output_paths['validation_parquet'], index=False)
    classification_df.to_parquet(output_paths['classification_parquet'], index=False)
    write_jsonl(rag_df.loc[rag_df['purpose'] == 'index'], output_paths['training_rag_jsonl'])
    write_jsonl(rag_df.loc[rag_df['purpose'] == 'evaluation'], output_paths['validation_rag_jsonl'])
    financial_df.head(1000).to_excel(output_paths['sample_excel'], index=False, sheet_name='financial_qa')
    output_paths['validation_report'].write_text(
        json.dumps(summary, ensure_ascii=False, indent=2, default=str) + '\n',
        encoding='utf-8',
    )
    issues_df.to_json(output_paths['issues'], orient='records', lines=True, force_ascii=False)

    print('저장 완료')
    for name, path in output_paths.items():
        print(f'- {name}: {path}')
else:
    print('SAVE_OUTPUTS=False: 메모리에만 DataFrame을 만들었습니다.')


## 다음 판단

1. 샘플 모드에서 열 이름·한글·결측·질문/정답 분리가 맞는지 확인합니다.
2. `RUN_MODE = 'full'`로 변경해 90,000행 전체 계약을 검증합니다.
3. 겹치는 `source_id` 3개를 기준으로 13개 QA 행의 격리 정책을 결정합니다.
4. 분류 모델은 `classification_df`의 Training으로 학습하고 Validation으로 평가합니다.
5. RAG는 `training_rag_index`만 색인하고 `validation_rag_evaluation`은 평가에만 사용합니다.
